# 2강 - 생성모형과 우도

이번 강의의 핵심 문장은 다음입니다.

```text
생성모형이 우도를 만들고,
사전분포와 우도를 곱한 뒤,
주변우도로 정규화하면 사후분포가 된다.
```

1강에서는 베이즈 갱신을 계산했습니다. 2강에서는 그 계산의 가운데 조각인 우도가 어디서 오는지 살펴봅니다.

## 0. 1강에서 이어지는 질문

노트북을 계속 진행하기 전에 대화창에서 다음을 확인합니다.

1. \(Y\mid\theta\sim Binomial(n,\theta)\)에서 \(Y\), \(n\), \(\theta\)는 각각 무엇인가요?
2. 관측값이 \(y=12\)라면 우도 \(L(\theta;y)\)는 무엇을 고정하고 무엇을 움직이나요?
3. `n=20`, `y=12`일 때 성공 수와 실패 수는 각각 몇 명인가요?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import binom
from scipy.stats import beta as beta_dist

plt.rcParams["figure.figsize"] = (7, 4)

## 1. 생성모형: 데이터가 만들어지는 이야기

노동시장 예제를 생각합니다.

- \(\theta\): 어떤 구직자가 일정 기간 안에 재취업할 확률
- \(n\): 관측한 구직자 수
- \(Y\): 관측 전에는 알 수 없는 재취업 성공자 수
- \(y\): 실제 관측된 재취업 성공자 수

생성모형은 다음처럼 쓸 수 있습니다.

\[
\theta \sim Beta(\alpha, \beta)
\]

\[
Y\mid\theta \sim Binomial(n,\theta)
\]

\[
y\ \text{observed}
\]

즉, 먼저 모수 \(\theta\)에 대한 불확실성이 있고, 그 \(\theta\)가 주어지면 데이터 \(Y\)가 생성된다고 가정합니다.

### 대화 체크포인트 1

노동시장 예제에서 \(\theta\), \(Y\), \(y\), \(n\)을 각각 한 문장으로 정의해보세요.

## 2. 우도 곡선: 같은 비율, 다른 표본 수

두 지역의 관측 재취업률이 모두 0.6이라고 합시다.

```text
A 지역: n = 20,  y = 12
B 지역: n = 200, y = 120
```

두 지역의 관측비율은 같지만, 표본 수는 다릅니다.

코드를 실행하기 전에 예측하세요.

- 두 우도 곡선의 꼭대기는 어디쯤일까요?
- 어느 곡선이 더 좁을까요?

In [ ]:
theta_grid = np.linspace(0.001, 0.999, 500)

lik_A = binom.pmf(12, 20, theta_grid)
lik_B = binom.pmf(120, 200, theta_grid)

# 폭을 비교하기 쉽도록 각 우도를 자기 최댓값으로 나눈 상대우도도 계산합니다.
rel_lik_A = lik_A / lik_A.max()
rel_lik_B = lik_B / lik_B.max()

plt.plot(theta_grid, rel_lik_A, label="A: n=20, y=12")
plt.plot(theta_grid, rel_lik_B, label="B: n=200, y=120")
plt.xlabel("theta")
plt.ylabel("Relative likelihood")
plt.legend()
plt.show()

theta_A_mle = theta_grid[np.argmax(lik_A)]
theta_B_mle = theta_grid[np.argmax(lik_B)]

print("A likelihood peak near:", round(theta_A_mle, 3))
print("B likelihood peak near:", round(theta_B_mle, 3))

In [ ]:
# 자동 점검 1: 우도 곡선의 위치와 형태
assert theta_grid.shape == lik_A.shape == lik_B.shape == rel_lik_A.shape == rel_lik_B.shape
assert abs(theta_A_mle - 0.6) < 0.01
assert abs(theta_B_mle - 0.6) < 0.01
idx_05 = np.argmin(np.abs(theta_grid - 0.5))
assert rel_lik_B[idx_05] < rel_lik_A[idx_05]

print("자동 점검 1 통과: 같은 관측비율에서 두 우도 곡선의 꼭대기가 0.6 근처에 있습니다.")

주의할 점이 있습니다. `lik_A.sum()`이나 `lik_B.sum()`이 1일 필요는 없습니다. 우도는 \(\theta\)에 대한 확률분포가 아니라, 관측값을 고정하고 \(\theta\)를 움직이며 보는 함수입니다.

In [ ]:
print("sum(lik_A):", lik_A.sum())
print("sum(lik_B):", lik_B.sum())

## 3. 정규화 상수와 주변우도

이번에는 유한한 후보 모수만 있다고 해봅시다.

```text
theta candidates = [0.2, 0.5, 0.8]
prior = [0.2, 0.5, 0.3]
n = 5, y = 4
```

정규화되지 않은 사후값은 다음입니다.

\[
P(Y=y\mid\theta)P(\theta)
\]

이 값들을 모두 더한 것이 주변우도 또는 evidence입니다.

\[
P(Y=y)=\sum_\theta P(Y=y\mid\theta)P(\theta)
\]

In [ ]:
theta_candidates = np.array([0.2, 0.5, 0.8])
prior = np.array([0.2, 0.5, 0.3])
n = 5
y = 4

likelihood = binom.pmf(y, n, theta_candidates)
unnormalized = prior * likelihood
evidence = unnormalized.sum()
posterior = unnormalized / evidence

table = pd.DataFrame({
    "theta": theta_candidates,
    "prior": prior,
    "likelihood": likelihood,
    "prior_x_likelihood": unnormalized,
    "posterior": posterior,
})

print("evidence:", evidence)
print("posterior sum:", posterior.sum())
table

### 대화 체크포인트 2

`evidence`는 왜 그냥 나누는 숫자가 아니라 \(P(Y=y)\)라고 부를 수 있을까요?

In [ ]:
# 자동 점검 2: 유한 가설 갱신
assert np.isclose(prior.sum(), 1)
assert np.isclose(posterior.sum(), 1)
assert evidence > 0
assert posterior[np.argmax(posterior)] == posterior[-1]

print("자동 점검 2 통과: 유한 가설에서 정규화된 사후확률을 계산했습니다.")

## 4. 격자 근사: 연속형 모수를 계산하기

연속형 \(\theta\)의 모든 값을 실제로 계산할 수는 없으므로, 촘촘한 격자를 만듭니다.

이번에는 다음 모델을 사용합니다.

\[
\theta\sim Beta(2,2)
\]

\[
Y\mid\theta\sim Binomial(20,\theta)
\]

\[
y=12
\]

연속형에서는 사후밀도의 합이 아니라 적분값이 1이 되어야 합니다.

In [ ]:
# 직접 입력 1: 격자 사후밀도 계산
# 아래 None을 적절한 코드로 바꾸세요.

theta_grid_2 = np.linspace(0.001, 0.999, 1000)
alpha_prior = 2
beta_prior = 2
n_obs = 20
y_obs = 12

prior_density = None
likelihood_grid = None
unnormalized_grid = None
evidence_grid = None
posterior_density = None

In [ ]:
# 자동 점검 3: 격자 사후밀도
assert prior_density is not None, "prior_density를 계산하세요."
assert likelihood_grid is not None, "likelihood_grid를 계산하세요."
assert posterior_density is not None, "posterior_density를 계산하세요."
assert theta_grid_2.shape == prior_density.shape == likelihood_grid.shape == posterior_density.shape
assert evidence_grid > 0

posterior_area = np.trapezoid(posterior_density, theta_grid_2)
assert np.isclose(posterior_area, 1, atol=1e-3)

print("자동 점검 3 통과: 격자 사후밀도의 적분값이 1에 가깝습니다.")

In [ ]:
plt.plot(theta_grid_2, prior_density, label="Prior Beta(2,2)")
plt.plot(theta_grid_2, posterior_density, label="Grid posterior")
plt.xlabel("theta")
plt.ylabel("Density")
plt.legend()
plt.show()

## 5. Beta-Binomial 갱신

Beta 사전분포와 Binomial 표본모형을 결합하면 사후분포도 Beta 분포가 됩니다.

\[
\theta\sim Beta(\alpha,\beta)
\]

\[
Y\mid\theta\sim Binomial(n,\theta)
\]

\[
\theta\mid Y=y\sim Beta(\alpha+y,\beta+n-y)
\]

이번 함수는 일부러 `n, y`를 입력으로 받습니다. 성공 수는 `y`, 실패 수는 `n-y`입니다.

In [ ]:
# 직접 입력 2: Beta-Binomial 갱신 함수

def beta_binomial_update(alpha_prior, beta_prior, n, y):
    # successes와 failures를 먼저 명시하세요.
    successes = None
    failures = None
    
    alpha_post = None
    beta_post = None
    posterior_mean = None
    ci_low, ci_high = None, None
    
    return alpha_post, beta_post, posterior_mean, ci_low, ci_high

In [ ]:
# 자동 점검 4: Beta-Binomial 함수
test_A = beta_binomial_update(3, 7, 20, 12)
test_B = beta_binomial_update(3, 7, 200, 120)
test_C = beta_binomial_update(2, 2, 5, 3)

assert test_A[0] == 15 and test_A[1] == 15
assert np.isclose(test_A[2], 0.5)
assert test_B[0] == 123 and test_B[1] == 87
assert np.isclose(test_B[2], 123 / 210)
assert test_C[0] == 5 and test_C[1] == 4
assert test_A[3] < test_A[2] < test_A[4]

print("자동 점검 4 통과: beta_binomial_update 함수가 여러 입력에서 동작합니다.")

## 6. 순차 갱신

사후분포는 다음 관측의 사전분포로 사용할 수 있습니다.

초기 사전분포가 `Beta(2, 2)`라고 합시다.

```text
데이터 1: n=5,  y=3
데이터 2: n=10, y=5
```

한 번에 갱신한 결과와 순차적으로 갱신한 결과가 같은지 확인합니다.

In [ ]:
# 직접 입력 3: 한 번에 갱신과 순차 갱신 비교

# 한 번에 갱신: 전체 성공 8, 전체 실패 7이므로 n=15, y=8
one_step = None

# 순차 갱신
after_first = None
after_second = None

print("one step:", one_step)
print("after first:", after_first)
print("after second:", after_second)

In [ ]:
# 자동 점검 5: 순차 갱신
assert one_step is not None
assert after_second is not None
assert one_step[0] == after_second[0]
assert one_step[1] == after_second[1]
assert np.isclose(one_step[2], after_second[2])

print("자동 점검 5 통과: 순차 갱신과 한 번에 갱신이 같은 사후분포를 줍니다.")

# 종합평가

객관식과 주관식은 대화창에서 한 문제씩 진행합니다. 프로그래밍 평가는 아래 셀을 사용하고, 코드와 실행 결과를 대화창에 제출하세요.

## 종합평가 - 객관식과 주관식

여기서 노트북을 멈추고 튜터의 질문에 답합니다.

## 프로그래밍 평가 1: 격자 사후분포 함수

다음 함수를 작성하세요.

요구사항:

- `binom.pmf(y, n, theta_grid)`로 우도를 계산합니다.
- `prior_density * likelihood`를 계산합니다.
- `np.trapezoid(..., theta_grid)`로 정규화합니다.
- `likelihood`, `posterior_density`를 반환합니다.

In [ ]:
def grid_posterior(theta_grid, prior_density, n, y):
    # 여기에 코드를 작성하세요.
    pass

In [ ]:
# 프로그래밍 평가 1 자동 점검
eval_grid = np.linspace(0.001, 0.999, 1000)
eval_prior = beta_dist(2, 2).pdf(eval_grid)
eval_likelihood, eval_posterior = grid_posterior(eval_grid, eval_prior, 20, 12)

assert eval_likelihood.shape == eval_grid.shape
assert eval_posterior.shape == eval_grid.shape
assert np.isclose(np.trapezoid(eval_posterior, eval_grid), 1, atol=1e-3)

eval_prior_2 = beta_dist(3, 7).pdf(eval_grid)
_, eval_posterior_2 = grid_posterior(eval_grid, eval_prior_2, 50, 20)
assert np.isclose(np.trapezoid(eval_posterior_2, eval_grid), 1, atol=1e-3)

print("프로그래밍 평가 1 통과: grid_posterior 함수가 새로운 입력에서도 동작합니다.")

## 프로그래밍 평가 2: Beta-Binomial 함수 재작성

아래 함수는 위에서 만든 함수와 같은 역할을 하지만, 종합평가에서는 처음부터 다시 작성합니다.

In [ ]:
def beta_binomial_summary(alpha_prior, beta_prior, n, y):
    # 여기에 코드를 작성하세요.
    pass

In [ ]:
# 프로그래밍 평가 2 자동 점검
summary_1 = beta_binomial_summary(2, 3, 5, 4)
summary_2 = beta_binomial_summary(3, 7, 20, 12)
summary_3 = beta_binomial_summary(1, 1, 10, 0)

assert summary_1[0] == 6 and summary_1[1] == 4
assert np.isclose(summary_1[2], 0.6)
assert summary_2[0] == 15 and summary_2[1] == 15
assert summary_3[0] == 1 and summary_3[1] == 11
assert summary_2[3] < summary_2[2] < summary_2[4]

print("프로그래밍 평가 2 통과: beta_binomial_summary 함수가 새로운 입력에서도 동작합니다.")

## 프로그래밍 평가 3: 노동시장 비교

공통 사전분포는 `Beta(3, 7)`입니다.

```text
A 지역: n=20,  y=12
B 지역: n=200, y=120
```

코드 실행 전 예측하세요.

- 어느 지역의 사후평균이 0.6에 더 가까울까요?
- 어느 지역의 사후분포가 더 좁을까요?

그다음 아래 요구사항을 만족하는 코드를 작성하세요.

1. `beta_binomial_summary`를 사용해 두 지역의 사후분포를 계산합니다.
2. 사후평균과 95% credible interval을 표로 출력합니다.
3. 두 사후밀도를 하나의 그래프에 그립니다.
4. 결과를 3~5문장으로 해석합니다.

In [ ]:
# 여기에 프로그래밍 평가 3 코드를 작성하세요.
